# REQ_117 Phase 2 — Parameter DMD Signal Check

Cheap inspection point for the windowed + per-regime parameter DMD analyzer.
Phase 2b (renderers + dashboard) is conditional on what these plots show.

## What we're looking for

Parameter DMD operates on per-(group, matrix) weight trajectories where
groups come from `neuron_grouping` at a configurable reference epoch and
`matrix ∈ {W_in, W_out}` are treated as separate sites — they may
reorganize on different timelines.

**The hypothesis we're checking against the activation-side narrative:**
the residual lens that worked at the activation centroid level should
also work at the weight level, with potentially *different* timing
between W_in and W_out reorganization (the empirical disparity that
motivated treating them as separate matrices).

## Reference variants

Same four as phase 1:

| Variant | Role |
|---|---|
| `p113/s999/ds598` | Canon — 4 frequency groups {9, 33, 38, 55} |
| `p109/s485/ds598` | Clean grokker |
| `p101/s999/ds598` | Degenerate / multi-segment orthogonal drift |
| `p59/s485/ds598`  | Failure |

For modadd, group ID corresponds to the dominant Fourier basis component:
`group_id = k - 1` where `k` is the frequency (canon group 8 → k=9, etc.).

## Exports

Each plot is saved to `apps/research/exports/req_117_parameter/{name}.png`.

## Setup

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

from miscope import load_family

MATRICES = ['W_in', 'W_out']
MATRIX_COLORS = {
    'W_in':  '#1f77b4',  # blue
    'W_out': '#d62728',  # red
}
VARIANT_COLORS = {
    'p113/s999/ds598': '#1f77b4',
    'p109/s485/ds598': '#2ca02c',
    'p101/s999/ds598': '#9467bd',
    'p59/s485/ds598':  '#8c564b',
}

VARIANTS = [
    {'label': 'p113/s999/ds598', 'role': 'canon',
     'parameters': {'prime': 113, 'seed': 999, 'data_seed': 598}},
    {'label': 'p109/s485/ds598', 'role': 'clean grokker',
     'parameters': {'prime': 109, 'seed': 485, 'data_seed': 598}},
    {'label': 'p101/s999/ds598', 'role': 'degenerate',
     'parameters': {'prime': 101, 'seed': 999, 'data_seed': 598}},
    {'label': 'p59/s485/ds598',  'role': 'failure',
     'parameters': {'prime': 59,  'seed': 485, 'data_seed': 598}},
]
ALL_LABELS = [v['label'] for v in VARIANTS]
FOCUS = 'p113/s999/ds598'

EXPORT_DIR = Path('../exports/req_117_parameter')
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

def show_and_export(fig, name):
    """Save figure to EXPORT_DIR/name.png and return for inline display."""
    fig.write_image(str(EXPORT_DIR / f'{name}.png'), scale=2)
    return fig


def group_label(group_id, prime):
    """For modadd, group_id maps to dominant frequency k = group_id + 1."""
    return f'group {int(group_id)} (k={int(group_id)+1})'

In [ ]:
family = load_family('modulo_addition_1layer')

def _get_variant(label):
    spec = next(v for v in VARIANTS if v['label'] == label)
    p = spec['parameters']
    return family.get_variant(prime=p['prime'], seed=p['seed'], data_seed=p['data_seed'])

def load_parameter_dmd(label):
    return _get_variant(label).artifacts.load_cross_epoch('parameter_dmd')

def load_activation_dmd(label):
    return _get_variant(label).artifacts.load_cross_epoch('activation_dmd')

data = {}
act_data = {}
for v in VARIANTS:
    try:
        data[v['label']] = load_parameter_dmd(v['label'])
    except FileNotFoundError as e:
        print(f'!! parameter_dmd {v["label"]}: {e}')
    try:
        act_data[v['label']] = load_activation_dmd(v['label'])
    except FileNotFoundError as e:
        print(f'!! activation_dmd {v["label"]}: {e}')

# Quick summary
for label, d in data.items():
    populated = d['populated_groups']
    n_per = d['group_n_neurons']
    ref = int(d['reference_epoch'])
    print(f'{label}: reference_epoch={ref}, {len(populated)} populated groups')
    for g, n in zip(populated, n_per):
        print(f'  group {int(g):>3} (k={int(g)+1:>3}): {int(n)} neurons')

## 1. Per-variant residual + regime boundaries

Rows = populated groups (in canonical order); columns = (W_in, W_out).
Residual is the per-window mean residual norm; dashed vertical lines
are detected regime boundaries; dotted horizontal line is the
boundary-detection threshold for that group/matrix.

**What to check:**
- Do W_in and W_out show different timing within a group? If yes, the
  decision to treat them as separate matrices is justified by the data.
- Do regime boundaries align with the second-descent / grokking window?
- Does any group show qualitatively different dynamics from its peers?

In [ ]:
def plot_residuals_with_regimes(d, label):
    populated = d['populated_groups']
    n_per = d['group_n_neurons']
    n_groups = len(populated)
    epochs = d['epochs']

    titles = []
    for g, n in zip(populated, n_per):
        titles.extend([
            f'{group_label(g, None)} — {n} neurons — W_in',
            f'{group_label(g, None)} — {n} neurons — W_out',
        ])

    fig = make_subplots(
        rows=n_groups, cols=2,
        shared_xaxes=True,
        vertical_spacing=0.05,
        horizontal_spacing=0.08,
        subplot_titles=titles,
    )

    for r, g in enumerate(populated, start=1):
        for c, matrix in enumerate(MATRICES, start=1):
            prefix = f'group_{int(g)}__{matrix}'
            starts = d[f'{prefix}__windowed__window_starts']
            ends = d[f'{prefix}__windowed__window_ends']
            centers = (starts + ends) // 2
            x = epochs[centers]
            y = d[f'{prefix}__windowed__residual_norm_mean']
            threshold = float(d[f'{prefix}__regimes__threshold_used'])
            boundaries = d[f'{prefix}__regimes__boundary_indices']

            fig.add_trace(
                go.Scatter(x=x, y=y, mode='lines',
                           line=dict(color=MATRIX_COLORS[matrix], width=1.5),
                           showlegend=False),
                row=r, col=c,
            )
            fig.add_hline(y=threshold, line_dash='dot', line_color='gray',
                          row=r, col=c)
            for b in boundaries:
                b_epoch = int(epochs[centers[int(b)]])
                fig.add_vline(x=b_epoch, line_dash='dash', line_color='black',
                              opacity=0.5, row=r, col=c)
            fig.update_yaxes(title_text='residual', row=r, col=c)

    for c in (1, 2):
        fig.update_xaxes(title_text='epoch', row=n_groups, col=c)
    fig.update_layout(
        title=f'{label} — Parameter DMD residuals + regime boundaries (rows=groups, cols=W_in/W_out)',
        height=max(300 * n_groups, 600),
        width=1100,
    )
    return fig

show_and_export(plot_residuals_with_regimes(data[FOCUS], FOCUS), 'residuals_p113')

In [ ]:
show_and_export(plot_residuals_with_regimes(data['p109/s485/ds598'], 'p109/s485/ds598'), 'residuals_p109')

In [ ]:
show_and_export(plot_residuals_with_regimes(data['p101/s999/ds598'], 'p101/s999/ds598'), 'residuals_p101')

In [ ]:
show_and_export(plot_residuals_with_regimes(data['p59/s485/ds598'], 'p59/s485/ds598'), 'residuals_p59')

## 2. Per-variant eigenvalue migration

Rows = populated groups; columns = (W_in, W_out). Each panel auto-zooms
to its actual eigenvalue cluster. Color encodes training time.

**What to check:**
- Per-group, do W_in and W_out produce the same eigenvalue character
  or distinct shapes?
- Across groups within a variant, are signatures consistent (the model
  is learning the same kind of dynamics across frequencies) or distinct
  (different frequencies have different parameter dynamics)?
- Compared to activation_dmd's pendant/fan/teardrop/sparse-line per-variant
  signatures, does the parameter side show *new* shapes?

In [ ]:
def plot_eigenvalue_migration(d, label):
    populated = d['populated_groups']
    n_per = d['group_n_neurons']
    n_groups = len(populated)
    epochs = d['epochs']
    theta = np.linspace(0, 2*np.pi, 200)
    cx, cy = np.cos(theta), np.sin(theta)

    titles = []
    for g, n in zip(populated, n_per):
        titles.extend([
            f'{group_label(g, None)} — {n} neurons — W_in',
            f'{group_label(g, None)} — {n} neurons — W_out',
        ])
    fig = make_subplots(
        rows=n_groups, cols=2,
        subplot_titles=titles,
        horizontal_spacing=0.10, vertical_spacing=0.10,
    )

    cell_idx = 0
    for r, g in enumerate(populated, start=1):
        for c, matrix in enumerate(MATRICES, start=1):
            prefix = f'group_{int(g)}__{matrix}'
            eigs = d[f'{prefix}__windowed__eigenvalues']
            n_modes_per_window = d[f'{prefix}__windowed__n_modes_per_window']
            starts = d[f'{prefix}__windowed__window_starts']
            ends = d[f'{prefix}__windowed__window_ends']
            center_epochs = epochs[(starts + ends) // 2]
            all_real, all_imag, all_epoch = [], [], []
            for w_idx in range(len(starts)):
                k = int(n_modes_per_window[w_idx])
                valid = eigs[w_idx, :k]
                all_real.extend(valid.real.tolist())
                all_imag.extend(valid.imag.tolist())
                all_epoch.extend([int(center_epochs[w_idx])] * k)

            if all_real:
                r_lo, r_hi = min(all_real), max(all_real)
                i_lo, i_hi = min(all_imag), max(all_imag)
                r_pad = max((r_hi - r_lo) * 0.10, 0.05)
                i_pad = max((i_hi - i_lo) * 0.10, 0.05)
                x_range = [r_lo - r_pad, r_hi + r_pad]
                y_range = [i_lo - i_pad, i_hi + i_pad]
            else:
                x_range = [-1.5, 1.5]
                y_range = [-1.5, 1.5]

            fig.add_trace(
                go.Scatter(x=cx, y=cy, mode='lines',
                           line=dict(color='lightgray', width=1),
                           hoverinfo='skip', showlegend=False, cliponaxis=True),
                row=r, col=c,
            )
            fig.add_trace(
                go.Scatter(
                    x=all_real, y=all_imag, mode='markers',
                    marker=dict(
                        size=4, color=all_epoch, colorscale='Viridis',
                        showscale=(cell_idx == 0),
                        colorbar=dict(title='epoch', len=0.4) if cell_idx == 0 else None,
                    ),
                    showlegend=False, cliponaxis=True,
                ),
                row=r, col=c,
            )
            fig.update_xaxes(title_text='Re(λ)', row=r, col=c, range=x_range)
            fig.update_yaxes(title_text='Im(λ)', row=r, col=c, range=y_range)
            cell_idx += 1

    fig.update_layout(
        title=f'{label} — Parameter DMD eigenvalue migration (rows=groups, cols=W_in/W_out)',
        height=max(380 * n_groups, 600),
        width=1100,
    )
    return fig

show_and_export(plot_eigenvalue_migration(data[FOCUS], FOCUS), 'eigenvalues_p113')

In [ ]:
show_and_export(plot_eigenvalue_migration(data['p109/s485/ds598'], 'p109/s485/ds598'), 'eigenvalues_p109')

In [ ]:
show_and_export(plot_eigenvalue_migration(data['p101/s999/ds598'], 'p101/s999/ds598'), 'eigenvalues_p101')

In [ ]:
show_and_export(plot_eigenvalue_migration(data['p59/s485/ds598'], 'p59/s485/ds598'), 'eigenvalues_p59')

## 3. Per-regime DMD residual vs. windowed

Same comparison as the activation_dmd version: the per-regime residuals
(black dashed bands) should generally be lower than the windowed signal
where the regime detection has captured something real.

In [ ]:
def plot_per_regime_vs_windowed(d, label):
    populated = d['populated_groups']
    n_per = d['group_n_neurons']
    n_groups = len(populated)
    epochs = d['epochs']
    titles = []
    for g, n in zip(populated, n_per):
        titles.extend([
            f'{group_label(g, None)} — {n} neurons — W_in',
            f'{group_label(g, None)} — {n} neurons — W_out',
        ])
    fig = make_subplots(
        rows=n_groups, cols=2,
        subplot_titles=titles,
        horizontal_spacing=0.10, vertical_spacing=0.08,
    )
    for r, g in enumerate(populated, start=1):
        for c, matrix in enumerate(MATRICES, start=1):
            prefix = f'group_{int(g)}__{matrix}'
            starts = d[f'{prefix}__windowed__window_starts']
            ends = d[f'{prefix}__windowed__window_ends']
            x = epochs[(starts + ends) // 2]
            y = d[f'{prefix}__windowed__residual_norm_mean']
            pr_starts = d[f'{prefix}__per_regime__segment_starts']
            pr_ends = d[f'{prefix}__per_regime__segment_ends']
            pr_residual = d[f'{prefix}__per_regime__residual_norm_mean']
            fig.add_trace(
                go.Scatter(x=x, y=y, mode='lines',
                           line=dict(color=MATRIX_COLORS[matrix], width=1),
                           showlegend=False),
                row=r, col=c,
            )
            for s, e, res in zip(pr_starts, pr_ends, pr_residual):
                if np.isnan(res):
                    continue
                x_seg = [int(epochs[s]), int(epochs[min(e - 1, len(epochs) - 1)])]
                fig.add_trace(
                    go.Scatter(x=x_seg, y=[res, res], mode='lines',
                               line=dict(color='black', width=2, dash='dash'),
                               showlegend=False),
                    row=r, col=c,
                )
            fig.update_yaxes(title_text='residual', type='log', row=r, col=c)
    for c in (1, 2):
        fig.update_xaxes(title_text='epoch', row=n_groups, col=c)
    fig.update_layout(
        title=f'{label} — Parameter DMD: windowed vs per-regime (log y)',
        height=max(300 * n_groups, 600),
        width=1100,
    )
    return fig

show_and_export(plot_per_regime_vs_windowed(data[FOCUS], FOCUS), 'per_regime_p113')

In [ ]:
show_and_export(plot_per_regime_vs_windowed(data['p109/s485/ds598'], 'p109/s485/ds598'), 'per_regime_p109')

In [ ]:
show_and_export(plot_per_regime_vs_windowed(data['p101/s999/ds598'], 'p101/s999/ds598'), 'per_regime_p101')

In [ ]:
show_and_export(plot_per_regime_vs_windowed(data['p59/s485/ds598'], 'p59/s485/ds598'), 'per_regime_p59')

## 4. W_in vs. W_out timing comparison (per group)

The decisive test for whether W_in and W_out really are independent
matrices: overlay their windowed residuals on the same axes per group.
If the peaks line up, the matrices are co-evolving and we could have
concatenated them. If they offset or differ in shape, separate treatment
is the right call.

In [ ]:
def plot_win_vs_wout_per_group(d, label):
    populated = d['populated_groups']
    n_per = d['group_n_neurons']
    n_groups = len(populated)
    epochs = d['epochs']
    titles = [f'{group_label(g, None)} — {n} neurons' for g, n in zip(populated, n_per)]
    fig = make_subplots(
        rows=n_groups, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.06,
        subplot_titles=titles,
    )
    for r, g in enumerate(populated, start=1):
        for matrix in MATRICES:
            prefix = f'group_{int(g)}__{matrix}'
            starts = d[f'{prefix}__windowed__window_starts']
            ends = d[f'{prefix}__windowed__window_ends']
            x = epochs[(starts + ends) // 2]
            y = d[f'{prefix}__windowed__residual_norm_mean']
            fig.add_trace(
                go.Scatter(x=x, y=y, mode='lines', name=matrix,
                           line=dict(color=MATRIX_COLORS[matrix], width=1.5),
                           legendgroup=matrix,
                           showlegend=(r == 1)),
                row=r, col=1,
            )
        fig.update_yaxes(title_text='residual', row=r, col=1)
    fig.update_xaxes(title_text='epoch', row=n_groups, col=1)
    fig.update_layout(
        title=f'{label} — W_in vs W_out residuals overlaid per group',
        height=max(280 * n_groups, 500),
        width=1100,
    )
    return fig

show_and_export(plot_win_vs_wout_per_group(data[FOCUS], FOCUS), 'win_vs_wout_p113')

In [ ]:
show_and_export(plot_win_vs_wout_per_group(data['p109/s485/ds598'], 'p109/s485/ds598'), 'win_vs_wout_p109')

In [ ]:
show_and_export(plot_win_vs_wout_per_group(data['p101/s999/ds598'], 'p101/s999/ds598'), 'win_vs_wout_p101')

In [ ]:
show_and_export(plot_win_vs_wout_per_group(data['p59/s485/ds598'], 'p59/s485/ds598'), 'win_vs_wout_p59')

## 5. Overlay spot-check: activation_dmd `mlp_out` vs parameter_dmd per-group W_in/W_out

Each line is normalized to its own max so timing is the only signal
(amplitudes are uncomparable: activation residuals peak around ~25 for canon
while parameter residuals peak around ~1). One row per populated group;
three lines per panel: `mlp_out` (purple), parameter group's `W_in` (blue),
parameter group's `W_out` (red).

**What we're checking:** do parameter-side regime boundaries align with
activation-side regime boundaries on the same training events? If yes,
this is direct evidence that the residual lens converges across
representational projections — within-DMD multi-lens consistency.
Cases where they diverge would be especially informative.

In [ ]:
def plot_mlpout_vs_param_groups(label):
    p_d = data[label]
    a_d = act_data.get(label)
    if a_d is None:
        print(f'No activation_dmd data for {label}; skipping')
        return None

    populated = p_d['populated_groups']
    n_per = p_d['group_n_neurons']
    n_groups = len(populated)
    epochs = p_d['epochs']

    # mlp_out from activation_dmd
    a_starts = a_d['mlp_out__windowed__window_starts']
    a_ends = a_d['mlp_out__windowed__window_ends']
    a_centers = (a_starts + a_ends) // 2
    a_x = a_d['epochs'][a_centers]
    a_y = a_d['mlp_out__windowed__residual_norm_mean']
    a_max = float(a_y.max()) if a_y.max() > 0 else 1.0

    titles = [f'{group_label(g, None)} — {n} neurons' for g, n in zip(populated, n_per)]
    fig = make_subplots(
        rows=n_groups, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.06,
        subplot_titles=titles,
    )

    for r, g in enumerate(populated, start=1):
        # mlp_out — same line on every row, normalized
        fig.add_trace(
            go.Scatter(x=a_x, y=a_y / a_max, mode='lines', name='mlp_out (act)',
                       line=dict(color='#9467bd', width=1.5),
                       legendgroup='mlp_out',
                       showlegend=(r == 1)),
            row=r, col=1,
        )
        # W_in
        prefix = f'group_{int(g)}__W_in'
        starts = p_d[f'{prefix}__windowed__window_starts']
        ends = p_d[f'{prefix}__windowed__window_ends']
        x = epochs[(starts + ends) // 2]
        y = p_d[f'{prefix}__windowed__residual_norm_mean']
        y_max = float(y.max()) if y.max() > 0 else 1.0
        fig.add_trace(
            go.Scatter(x=x, y=y / y_max, mode='lines', name='W_in (param)',
                       line=dict(color=MATRIX_COLORS['W_in'], width=1.5),
                       legendgroup='W_in',
                       showlegend=(r == 1)),
            row=r, col=1,
        )
        # W_out
        prefix = f'group_{int(g)}__W_out'
        starts = p_d[f'{prefix}__windowed__window_starts']
        ends = p_d[f'{prefix}__windowed__window_ends']
        x = epochs[(starts + ends) // 2]
        y = p_d[f'{prefix}__windowed__residual_norm_mean']
        y_max = float(y.max()) if y.max() > 0 else 1.0
        fig.add_trace(
            go.Scatter(x=x, y=y / y_max, mode='lines', name='W_out (param)',
                       line=dict(color=MATRIX_COLORS['W_out'], width=1.5),
                       legendgroup='W_out',
                       showlegend=(r == 1)),
            row=r, col=1,
        )
        fig.update_yaxes(title_text='residual / max', row=r, col=1)

    fig.update_xaxes(title_text='epoch', row=n_groups, col=1)
    fig.update_layout(
        title=f'{label} — mlp_out vs per-group W_in/W_out (each max-normalized)',
        height=max(280 * n_groups, 500),
        width=1100,
    )
    return fig

show_and_export(plot_mlpout_vs_param_groups(FOCUS), 'overlay_p113')

In [ ]:
show_and_export(plot_mlpout_vs_param_groups('p109/s485/ds598'), 'overlay_p109')

In [ ]:
show_and_export(plot_mlpout_vs_param_groups('p101/s999/ds598'), 'overlay_p101')

In [ ]:
show_and_export(plot_mlpout_vs_param_groups('p59/s485/ds598'), 'overlay_p59')

## Observations (fill in after inspection)

### Per-variant
- p113 (canon): …
- p109 (clean grokker): …
- p101 (degenerate): …
- p59 (failure): …

### W_in vs W_out timing
- Are W_in / W_out residual peaks coincident or offset within a group?
  If offset, in which direction (does one consistently lead)?
- Is the offset character consistent across groups within a variant, or
  group-dependent?
- Cross-variant: do all four variants show the same W_in / W_out timing
  relationship, or is the relationship itself a per-variant signature?

### Comparison to activation_dmd
- Do parameter-side regime boundaries align with the activation-side
  boundaries we already documented? (Or do they offset, suggesting
  parameter dynamics lead/lag activation dynamics?)
- Do the per-variant eigenvalue signatures echo the activation-side
  signatures (pendant / fan / teardrop / sparse line) or produce new
  shapes?

### Decision
- [ ] Proceed to phase 2b (renderers + dashboard page)
- [ ] Refine analyzer (PCA cap, threshold, etc.) and re-run
- [ ] Stop